In [15]:
import torch
from torch import nn, optim
from torch.utils import data
import wandb
import math
import uuid

In [ ]:
# def train_one_epoch(model, loader, loss_fn, optimizer):
#     model.train()
#     for train_x, train_y in loader:
#         y_hat = model(train_x).reshape(-1, 1)
#         loss = loss_fn(y_hat, train_y)
#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()

In [23]:
# corrected version
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss, n = 0.0, 0
    for train_x, train_y in loader:
        y_hat = model(train_x)
        loss = loss_fn(y_hat, train_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * train_x.size(0)
        n += train_x.size(0)
    return total_loss / n

In [ ]:
# def evaluate(model, loader, loss_fn):
#     model.eval()
#     with torch.no_grad():
#         # 只做前向，累加 loss/正确数
#         loss = torch.tensor(0.0)
#         for eval_x, eval_y in loader:
#             y_hat = model(eval_x).reshape(-1, 1)
#             loss += loss_fn(y_hat, eval_y)
#         return loss

In [24]:
# the correct version
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, n = 0.0, 0
    with torch.no_grad():
        for eval_x, eval_y in loader:
            y_hat = model(eval_x)
            batch_loss = loss_fn(y_hat, eval_y)
            total_loss += batch_loss.item() * eval_x.size(0)
            n += eval_x.size(0)
        return total_loss / n

In [ ]:
# def train_loop(model, loader, loss_fn, optimizer):
#     for epoch in range(2000):
#         train_one_epoch(model, loader, loss_fn, optimizer)
#         loss = evaluate(model, loader, loss_fn)
#         print(f"epoch: {epoch}, loss: {loss:f}")
#         wandb.log({"loss": loss.item()})


In [25]:
# corrected version
def train_loop(model, train_loader, eval_loader, loss_fn, optimizer):
    for epoch in range(2000):
        train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer)
        eval_loss = evaluate(model, eval_loader, loss_fn)
        wandb.log({"epoch": epoch, "train_loss": train_loss, "eval_loss": eval_loss})

In [26]:
def batch_loader(data_array, batch_size, is_train=True):
    dataset = data.TensorDataset(*data_array)
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

In [27]:
X = torch.linspace(-math.pi, math.pi, 1000).reshape(-1, 1)
y = torch.sin(X).reshape(-1, 1)
X_train, X_eval = X[:800], X[800:]
y_train, y_eval = y[:800], y[800:]

In [32]:
model = nn.Sequential(
    nn.Linear(1, 64),
    nn.Tanh(),
    nn.Linear(64, 1)
)
loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
train_loader = batch_loader((X_train, y_train), batch_size=100, is_train=True)
eval_loader = batch_loader((X_eval, y_eval), batch_size=100, is_train=False)

In [33]:
if wandb.run is not None:
    wandb.finish()
# run_id = uuid.uuid4().hex[:8]
wandb.init(project="torchlearn-stage1", name=f"day08-train-loop-batch-100-lr-0.001")
train_loop(model, train_loader, eval_loader, loss_fn, optimizer)

epoch,▁▁▁▁▁▂▂▂▂▂▂▃▃▄▄▄▄▄▄▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇██████
eval_loss,█▆▅▄▃▂▂▂▂▂▂▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss,▂▁▁▁▁▁▁▁▁▁▆▁▁▁▃▃▁▁▁█▁▃▁▁▂▁▁▁▁▁▁▁▁▂▁▁▁▃▁▁
epoch,1999
eval_loss,0.05494
train_loss,8e-05
